In [1]:
!pip install -q transformers datasets peft bitsandbytes trl accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.1 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B"

# 1. Define the 4-bit quantization rules
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # float16 works best on Colab T4s
)

# 2. Load the Tokenizer and the squeezed Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Fix a common tokenizer quirk for training
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

In [3]:
from peft import LoraConfig,get_peft_model


lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model=get_peft_model(model,lora_config)

model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [12]:
from datasets import Dataset

# 5 examples of sloppy human text -> perfect JSON
raw_data = [
    {"input": "Bought two packs of gum at 7-Eleven for 1.99 each", "output": '{"store": "7-Eleven", "items": [{"name": "gum", "qty": 2, "unit_price": 1.99}]}'},
    {"input": "Paid $45.50 for an oil change at Jiffy Lube on Tuesday", "output": '{"store": "Jiffy Lube", "items": [{"name": "oil change", "qty": 1, "unit_price": 45.50}]}'},
    {"input": "Grabbed 3 coffees from Dunkin, total came out to 12.45", "output": '{"store": "Dunkin", "items": [{"name": "coffee", "qty": 3, "unit_price": 4.15}]}'},
    {"input": "Target run: diapers were 24.99 and wipes were 4.50.", "output": '{"store": "Target", "items": [{"name": "diapers", "qty": 1, "unit_price": 24.99}, {"name": "wipes", "qty": 1, "unit_price": 4.50}]}'},
    {"input": "Got a single taco at Taco Bell for 2.10", "output": '{"store": "Taco Bell", "items": [{"name": "taco", "qty": 1, "unit_price": 2.10}]}'}
]

# THE FIX: Notice the {tokenizer.eos_token} appended to the very end of the output.
# This explicitly teaches the model that generating the EOS token is the final step!
def format_prompt(row):
    prompt = f"Extract transaction details to JSON.\nInput: {row['input']}\nOutput: {row['output']}{tokenizer.eos_token}"
    return {"text": prompt}

dataset = Dataset.from_list(raw_data).map(format_prompt)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [13]:
from trl import SFTTrainer, SFTConfig

# In the latest versions of the `trl` library, layout parameters
# like the text field and max length are moved into SFTConfig
training_args = SFTConfig(
    output_dir="./lora_cashier",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    learning_rate=2e-4,
    logging_steps=5,
    optim="paged_adamw_8bit",
    save_strategy="no",

    # --- NEW PLACEMENT FOR THESE ARGUMENTS ---
    dataset_text_field="text",
    max_length=128
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,  # 'tokenizer' argument was renamed to 'processing_class'
)

# Kick off the training!
trainer.train()

/tmp/ipykernel_782/40179619.py:5: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Step,Training Loss
5,0.364665
10,0.270056
15,0.206318
20,0.135905
25,0.073527
30,0.041159
35,0.031682
40,0.029356


TrainOutput(global_step=40, training_loss=0.1440834581851959, metrics={'train_runtime': 76.6784, 'train_samples_per_second': 1.304, 'train_steps_per_second': 0.522, 'total_flos': 55059583733760.0, 'train_loss': 0.1440834581851959, 'epoch': 20.0})

In [14]:
# A totally unseen test prompt
test_input = "Extract transaction details to JSON.\nInput: Snagged 4 energy drinks at CVS for 11.20 total.\nOutput:"

inputs = tokenizer(test_input, return_tensors="pt").to("cuda")

# THE FIX: Switch the model to Evaluation mode to stop the gradient caching warnings
model.eval()

# THE FIX: Add explicit generation boundaries (temperature and eos_token_id)
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    use_cache=True,
    temperature=0.1,                 # Keep it focused, no creative rambling
    do_sample=False,                 # Greedy decoding for strict JSON
    eos_token_id=tokenizer.eos_token_id, # Force it to respect the stop token
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n--- THE MODEL'S WORK ---")
print(response)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- THE MODEL'S WORK ---
Extract transaction details to JSON.
Input: Snagged 4 energy drinks at CVS for 11.20 total.
Output: {"store": "CVS", "items": [{"name": "energy drinks", "qty": 4, "unit_price": 2.80}]}
